# Bloque 1 — Feature Engineering y Feature Store

Construye observaciones, features y training set point-in-time para el modelo de propensión de recompra de Andina Market. Publica features reutilizables en Unity Catalog y demuestra corrección temporal sin leakage.

**Supuestos**:
- Nivel 0 completado: datos en `andina_market_lhcl_prod.rdv_retail`
- Mes 13 excluido del entrenamiento (reservado para Bloque 5)
- Cortes mensuales: 2025-07-01 a 2026-03-01 (9 cortes)
- 12 features documentadas con ventana, fuente, tipo y valor por defecto

**Resultado esperado**:
- Tabla `customer_observations` con etiquetas validadas
- Tabla `customer_features` registrada como Feature Table con PK y TIMESERIES
- Training set construido con `FeatureLookup` point-in-time
- Prueba de leakage automatizada
- Ejemplo de cliente explicado paso a paso

In [0]:
%pip install databricks-feature-engineering mlflow -q

In [0]:
from pyspark.sql.functions import (
    col, count as spark_count, sum as spark_sum, avg as spark_avg,
    max as spark_max, min as spark_min, when, lit, datediff,
    date_add, date_sub, desc, row_number, countDistinct, coalesce,
)
from pyspark.sql.window import Window
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

# ── Unity Catalog names (fully qualified) ──
CATALOG = "andina_market_lhcl_prod"
SOURCE_SCHEMA = "rdv_retail"
FEATURES_SCHEMA = "features"
OBSERVATIONS_TABLE = f"{CATALOG}.{FEATURES_SCHEMA}.customer_observations"
FEATURES_TABLE = f"{CATALOG}.{FEATURES_SCHEMA}.customer_features"

# ── Cutoff dates: monthly 2025-07-01 to 2026-03-01 (9 cortes) ──
CUTOFF_DATES = [
    "2025-07-01", "2025-08-01", "2025-09-01", "2025-10-01",
    "2025-11-01", "2025-12-01", "2026-01-01", "2026-02-01", "2026-03-01",
]

# ── Feature specification (12 features) ──
FEATURE_SPECS = [
    {"name": "days_since_last_order",       "source": "orders",            "type": "int",    "window": "point-in-time", "default": 9999},
    {"name": "orders_count_30d",            "source": "orders",            "type": "int",    "window": "30d",          "default": 0},
    {"name": "orders_count_90d",            "source": "orders",            "type": "int",    "window": "90d",          "default": 0},
    {"name": "avg_ticket_value_90d",        "source": "orders",            "type": "double", "window": "90d",          "default": 0.0},
    {"name": "payment_failure_rate_90d",    "source": "payments",          "type": "double", "window": "90d",          "default": 0.0},
    {"name": "view_product_count_14d",      "source": "app_events",        "type": "int",    "window": "14d",          "default": 0},
    {"name": "search_count_14d",            "source": "app_events",        "type": "int",    "window": "14d",          "default": 0},
    {"name": "web_activity_count_14d",      "source": "app_events",        "type": "int",    "window": "14d",          "default": 0},
    {"name": "cart_abandonment_sessions_14d","source": "app_events (derived)","type":"int",  "window": "14d",          "default": 0},
    {"name": "favorite_product_sku",         "source": "order_items+orders", "type": "string", "window": "historical",   "default": "UNKNOWN"},
    {"name": "tenure_days",                  "source": "customers",         "type": "int",    "window": "point-in-time", "default": 0},
    {"name": "customer_segment",            "source": "customers",         "type": "string", "window": "static",       "default": "bronze"},
]

fe = FeatureEngineeringClient()
print(f"FeatureEngineeringClient initialized. {len(FEATURE_SPECS)} features defined.")

# ADR-001: Feature Table temporal persistida vs Feature Views declarativas

## Decisión

Implementar una Feature Table temporal persistida como Delta table en
Unity Catalog, con clave primaria compuesta por:

- customer_id
- feature_timestamp TIMESERIES

Las features serán calculadas mediante un proceso batch reproducible y
actualizadas de forma idempotente mediante MERGE.

## Contexto

Las features principales del modelo son agregaciones históricas sobre
ventanas de 14, 30 y 90 días. Debemos conservar snapshots por cliente y
fecha para construir training sets point-in-time correctos, inspeccionar
los valores históricos y demostrar ausencia de fuga de información.

## ADR-002: Features on-demand

**Decisión**: No implementar features on-demand en la primera versión.

**Razones**:
- Todas las features se pueden materializar por corte (`cutoff_ts`).
- `tenure_days` depende de `signup_at` y el timestamp de scoring, pero se materializa por corte para mantener la consistencia point-in-time.

**Alternativa descartada**: Calcular `days_since_last_order` on-demand en inferencia. Útil para serving real-time, pero innecesario para batch scoring.

**Consecuencia operativa**: En inferencia batch, se calculan features con la fecha de scoring como cutoff.

**Condición para reconsiderar**: Si se necesita serving en tiempo real, `days_since_last_order` sería la primera candidata on-demand, con una función gobernada disponible tanto en entrenamiento como en inferencia.

In [0]:
# Create schema for feature tables (idempotent)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{FEATURES_SCHEMA}")
spark.sql(f"COMMENT ON SCHEMA {CATALOG}.{FEATURES_SCHEMA} IS 'Feature tables and training observations for Andina Market repurchase model'")
print(f"Schema {CATALOG}.{FEATURES_SCHEMA} ready.")

## Paso 1: Observaciones y etiquetas

Para cada corte mensual `t`, identificamos clientes activos (con al menos una orden antes de `t`) y calculamos la etiqueta: ¿compró al menos una orden `completed` en `[t, t+30d)`?

In [0]:
# Build observations: one row per (customer_id, cutoff_ts) with label
# Label = 1 if at least one 'completed' order in [t, t+30d)
cutoff_str = ", ".join([f"timestamp('{d} 00:00:00')" for d in CUTOFF_DATES])

obs_df = spark.sql(f"""
WITH cutoffs AS (
  SELECT explode(array({cutoff_str})) AS cutoff_ts
),
all_customers AS (
  SELECT DISTINCT customer_id FROM {CATALOG}.{SOURCE_SCHEMA}.orders
),
obs_base AS (
  SELECT c.customer_id, ct.cutoff_ts
  FROM cutoffs ct
  CROSS JOIN all_customers c
),
order_signals AS (
  SELECT
    ob.customer_id,
    ob.cutoff_ts,
    COUNT(CASE WHEN o.order_ts < ob.cutoff_ts THEN 1 END) AS orders_before_cutoff,
    COUNT(CASE WHEN o.order_ts >= ob.cutoff_ts
               AND o.order_ts < date_add(ob.cutoff_ts, 30)
               AND o.status = 'completed' THEN 1 END) AS completed_in_window
  FROM obs_base ob
  LEFT JOIN {CATALOG}.{SOURCE_SCHEMA}.orders o
    ON o.customer_id = ob.customer_id
  GROUP BY ob.customer_id, ob.cutoff_ts
)
SELECT
  customer_id,
  cutoff_ts,
  CASE WHEN completed_in_window > 0 THEN 1 ELSE 0 END AS repurchased_next_30d
FROM order_signals
WHERE orders_before_cutoff > 0
ORDER BY customer_id, cutoff_ts
""")

obs_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OBSERVATIONS_TABLE)
print(f"Observations written to {OBSERVATIONS_TABLE}")
display(spark.table(OBSERVATIONS_TABLE).limit(10))

In [0]:
# ── Validate observations ──
obs = spark.table(OBSERVATIONS_TABLE)

# 1. Uniqueness of (customer_id, cutoff_ts)
dup_count = obs.groupBy("customer_id", "cutoff_ts").count().filter("count > 1").count()
assert dup_count == 0, f"FAIL: Duplicate (customer_id, cutoff_ts) pairs: {dup_count}"

# 2. Label only uses completed orders — spot check 5 positive and 5 negative
pos_samples = obs.filter("repurchased_next_30d = 1").limit(5).collect()
neg_samples = obs.filter("repurchased_next_30d = 0").limit(5).collect()
for row in pos_samples:
    has_completed = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {CATALOG}.{SOURCE_SCHEMA}.orders
        WHERE customer_id = '{row['customer_id']}'
          AND order_ts >= timestamp('{row['cutoff_ts']}')
          AND order_ts < date_add(timestamp('{row['cutoff_ts']}'), 30)
          AND status = 'completed'
    """).collect()[0]['cnt']
    assert has_completed > 0, f"FAIL: label=1 but no completed order for {row['customer_id']} at {row['cutoff_ts']}"
for row in neg_samples:
    has_completed = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {CATALOG}.{SOURCE_SCHEMA}.orders
        WHERE customer_id = '{row['customer_id']}'
          AND order_ts >= timestamp('{row['cutoff_ts']}')
          AND order_ts < date_add(timestamp('{row['cutoff_ts']}'), 30)
          AND status = 'completed'
    """).collect()[0]['cnt']
    assert has_completed == 0, f"FAIL: label=0 but found completed order for {row['customer_id']} at {row['cutoff_ts']}"

# 3. No month_13 data in cutoffs
max_cutoff = obs.agg(spark_max("cutoff_ts")).collect()[0][0]
assert str(max_cutoff).startswith("2026-03"), f"FAIL: unexpected max cutoff: {max_cutoff}"

# 4. Row count and label distribution
row_count = obs.count()
label_dist = obs.groupBy("repurchased_next_30d").count().orderBy("repurchased_next_30d").collect()
print(f"✅ Observations: {row_count} rows")
print(f"✅ Label distribution: {[(r['repurchased_next_30d'], r['count']) for r in label_dist]}")
print(f"✅ Duplicate (customer_id, cutoff_ts) pairs: {dup_count}")
print(f"✅ Max cutoff: {max_cutoff}")
print(f"✅ Label spot-check passed (5 positive, 5 negative verified)")

## Paso 2: Calcular 12 features por corte

Cada feature se calcula con eventos **estrictamente anteriores** al cutoff. Las fuentes se procesan por grupo para mantener el código legible y depurable.

**Reglas temporales** (per AGENTS.md):
- Órdenes: `order_ts < cutoff_ts`
- Pagos: `status_updated_at < cutoff_ts`
- App events: `event_ts < cutoff_ts`
- Tickets: `created_at < cutoff_ts AND (closed_at IS NULL OR closed_at >= cutoff_ts)`

### Especificación de features

| # | Feature | Fuente | Tipo | Ventana | Default | Descripción |
|---|---------|--------|------|---------|---------|-------------|
| 1 | `days_since_last_order` | orders | int | punto | 9999 | Días desde la última orden al corte |
| 2 | `orders_count_30d` | orders | int | 30d | 0 | Órdenes en los 30 días antes del corte |
| 3 | `orders_count_90d` | orders | int | 90d | 0 | Órdenes en los 90 días antes del corte |
| 4 | `avg_ticket_value_90d` | orders | double | 90d | 0.0 | Ticket promedio en 90 días |
| 5 | `payment_failure_rate_90d` | payments | double | 90d | 0.0 | Tasa de pagos failed+declined (usa status_updated_at < t) |
| 6 | `view_product_count_14d` | app_events | int | 14d | 0 | Vistas de producto en 14 días |
| 7 | `search_count_14d` | app_events | int | 14d | 0 | Búsquedas en 14 días |
| 8 | `web_activity_count_14d` | app_events | int | 14d | 0 | Eventos totales en 14 días |
| 9 | `cart_abandonment_sessions_14d` | app_events (derivada) | int | 14d | 0 | Sesiones con add_to_cart sin purchase |
| 10 | `favorite_product_sku` | order_items+orders | string | histórico | UNKNOWN | SKU más comprado antes del corte |
| 11 | `tenure_days` | customers | int | punto | 0 | Días desde el alta del cliente |
| 12 | `customer_segment` | customers | string | estática | bronze | Segmento: bronze/silver/gold |

In [0]:
# Features 1-4 from orders: days_since_last_order, orders_count_30d, orders_count_90d, avg_ticket_value_90d
orders_feats = spark.sql(f"""
SELECT
  obs.customer_id,
  obs.cutoff_ts AS feature_timestamp,
  COALESCE(
    DATEDIFF(obs.cutoff_ts, MAX(CASE WHEN o.order_ts < obs.cutoff_ts THEN o.order_ts END)),
    9999
  ) AS days_since_last_order,
  COUNT(CASE WHEN o.order_ts >= date_sub(obs.cutoff_ts, 30)
             AND o.order_ts < obs.cutoff_ts THEN 1 END) AS orders_count_30d,
  COUNT(CASE WHEN o.order_ts >= date_sub(obs.cutoff_ts, 90)
             AND o.order_ts < obs.cutoff_ts THEN 1 END) AS orders_count_90d,
  COALESCE(
    AVG(CASE WHEN o.order_ts >= date_sub(obs.cutoff_ts, 90)
             AND o.order_ts < obs.cutoff_ts THEN o.total_amount END),
    0.0
  ) AS avg_ticket_value_90d
FROM {OBSERVATIONS_TABLE} obs
LEFT JOIN {CATALOG}.{SOURCE_SCHEMA}.orders o
  ON o.customer_id = obs.customer_id
GROUP BY obs.customer_id, obs.cutoff_ts
""")
orders_feats.createOrReplaceTempView("orders_feats")
print(f"Orders features: {orders_feats.count()} rows")

In [0]:
# Feature 5 from payments: payment_failure_rate_90d
# Failures = status IN ('failed', 'declined'). Uses status_updated_at < cutoff (per AGENTS.md).
payments_feats = spark.sql(f"""
SELECT
  obs.customer_id,
  obs.cutoff_ts AS feature_timestamp,
  COALESCE(
    SUM(CASE WHEN p.status IN ('failed', 'declined') THEN 1 ELSE 0 END) / NULLIF(COUNT(p.payment_id), 0),
    0.0
  ) AS payment_failure_rate_90d
FROM {OBSERVATIONS_TABLE} obs
LEFT JOIN {CATALOG}.{SOURCE_SCHEMA}.payments p
  ON p.customer_id = obs.customer_id
  AND p.status_updated_at >= date_sub(obs.cutoff_ts, 90)
  AND p.status_updated_at < obs.cutoff_ts
GROUP BY obs.customer_id, obs.cutoff_ts
""")
payments_feats.createOrReplaceTempView("payments_feats")
print(f"Payments features: {payments_feats.count()} rows")

In [0]:
# Features 6-9 from app_events
# 6: view_product_count_14d, 7: search_count_14d, 8: web_activity_count_14d
# 9: cart_abandonment_sessions_14d (derived: sessions with add_to_cart but no purchase)
app_feats = spark.sql(f"""
WITH obs AS (
  SELECT customer_id, cutoff_ts FROM {OBSERVATIONS_TABLE}
),
event_counts AS (
  SELECT
    obs.customer_id,
    obs.cutoff_ts AS feature_timestamp,
    COUNT(CASE WHEN ae.event_type = 'view_product' THEN 1 END) AS view_product_count_14d,
    COUNT(CASE WHEN ae.event_type = 'search' THEN 1 END) AS search_count_14d,
    COUNT(ae.event_id) AS web_activity_count_14d
  FROM obs
  LEFT JOIN {CATALOG}.{SOURCE_SCHEMA}.app_events ae
    ON ae.customer_id = obs.customer_id
    AND ae.event_ts >= date_sub(obs.cutoff_ts, 14)
    AND ae.event_ts < obs.cutoff_ts
  GROUP BY obs.customer_id, obs.cutoff_ts
),
session_signals AS (
  SELECT
    obs.customer_id,
    obs.cutoff_ts,
    ae.session_id,
    MAX(CASE WHEN ae.event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS has_add_to_cart,
    MAX(CASE WHEN ae.event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
  FROM obs
  JOIN {CATALOG}.{SOURCE_SCHEMA}.app_events ae
    ON ae.customer_id = obs.customer_id
    AND ae.event_ts >= date_sub(obs.cutoff_ts, 14)
    AND ae.event_ts < obs.cutoff_ts
  GROUP BY obs.customer_id, obs.cutoff_ts, ae.session_id
),
cart_abandon AS (
  SELECT
    customer_id,
    cutoff_ts AS feature_timestamp,
    COUNT(DISTINCT CASE WHEN has_add_to_cart = 1 AND has_purchase = 0 THEN session_id END) AS cart_abandonment_sessions_14d
  FROM session_signals
  GROUP BY customer_id, cutoff_ts
)
SELECT
  ec.customer_id,
  ec.feature_timestamp,
  ec.view_product_count_14d,
  ec.search_count_14d,
  ec.web_activity_count_14d,
  COALESCE(ca.cart_abandonment_sessions_14d, 0) AS cart_abandonment_sessions_14d
FROM event_counts ec
LEFT JOIN cart_abandon ca
  ON ca.customer_id = ec.customer_id
  AND ca.feature_timestamp = ec.feature_timestamp
""")
app_feats.createOrReplaceTempView("app_feats")
print(f"App events features: {app_feats.count()} rows")

In [0]:
# Features 10-12 from customers and order_items
# 10: favorite_product_sku (most purchased SKU before cutoff, via window function)
# 11: tenure_days (DATEDIFF(cutoff_ts, signup_at))
# 12: customer_segment (from customers.segment)

# Feature 10: favorite_product_sku
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW fav_sku AS
WITH item_counts AS (
  SELECT
    o.customer_id,
    obs.cutoff_ts,
    oi.sku,
    COUNT(*) AS sku_count
  FROM {OBSERVATIONS_TABLE} obs
  JOIN {CATALOG}.{SOURCE_SCHEMA}.orders o
    ON o.customer_id = obs.customer_id AND o.order_ts < obs.cutoff_ts
  JOIN {CATALOG}.{SOURCE_SCHEMA}.order_items oi
    ON oi.order_id = o.order_id
  GROUP BY o.customer_id, obs.cutoff_ts, oi.sku
),
ranked AS (
  SELECT customer_id, cutoff_ts, sku,
    ROW_NUMBER() OVER (PARTITION BY customer_id, cutoff_ts ORDER BY sku_count DESC, sku ASC) AS rn
  FROM item_counts
)
SELECT customer_id, cutoff_ts AS feature_timestamp, sku AS favorite_product_sku
FROM ranked WHERE rn = 1
""")

# Features 11-12 + merge with favorite_product_sku
customer_feats = spark.sql(f"""
WITH base AS (
  SELECT
    obs.customer_id,
    obs.cutoff_ts AS feature_timestamp,
    COALESCE(DATEDIFF(obs.cutoff_ts, c.signup_at), 0) AS tenure_days,
    COALESCE(c.segment, 'bronze') AS customer_segment
  FROM {OBSERVATIONS_TABLE} obs
  LEFT JOIN {CATALOG}.{SOURCE_SCHEMA}.customers c
    ON c.customer_id = obs.customer_id
)
SELECT
  b.customer_id,
  b.feature_timestamp,
  COALESCE(fs.favorite_product_sku, 'UNKNOWN') AS favorite_product_sku,
  b.tenure_days,
  b.customer_segment
FROM base b
LEFT JOIN fav_sku fs
  ON fs.customer_id = b.customer_id AND fs.feature_timestamp = b.feature_timestamp
""")
customer_feats.createOrReplaceTempView("customer_feats")
print(f"Customer features: {customer_feats.count()} rows")

In [0]:
# Combine all 12 features into one DataFrame
all_features = spark.sql("""
SELECT
  o.customer_id,
  o.feature_timestamp,
  o.days_since_last_order,
  o.orders_count_30d,
  o.orders_count_90d,
  o.avg_ticket_value_90d,
  p.payment_failure_rate_90d,
  a.view_product_count_14d,
  a.search_count_14d,
  a.web_activity_count_14d,
  a.cart_abandonment_sessions_14d,
  c.favorite_product_sku,
  c.tenure_days,
  c.customer_segment
FROM orders_feats o
LEFT JOIN payments_feats p
  ON p.customer_id = o.customer_id AND p.feature_timestamp = o.feature_timestamp
LEFT JOIN app_feats a
  ON a.customer_id = o.customer_id AND a.feature_timestamp = o.feature_timestamp
LEFT JOIN customer_feats c
  ON c.customer_id = o.customer_id AND c.feature_timestamp = o.feature_timestamp
""")
all_features.createOrReplaceTempView("all_features")
print(f"Combined features: {all_features.count()} rows, {len(all_features.columns)} columns")
display(all_features.limit(10))

In [0]:
# ── Validate features ──
# 1. Uniqueness of (customer_id, feature_timestamp)
dup_count = all_features.groupBy("customer_id", "feature_timestamp").count().filter("count > 1").count()
assert dup_count == 0, f"FAIL: Duplicate (customer_id, feature_timestamp) pairs: {dup_count}"

# 2. Row count matches observations
obs_count = spark.table(OBSERVATIONS_TABLE).count()
feat_count = all_features.count()
assert obs_count == feat_count, f"FAIL: obs={obs_count} != features={feat_count}"

# 3. No negative recency (leakage indicator)
negative_recency = all_features.filter("days_since_last_order < 0").count()
assert negative_recency == 0, f"FAIL: {negative_recency} rows with negative days_since_last_order"

# 4. Null counts per column
null_counts = all_features.select([
    spark_count(when(col(c).isNull(), 1)).alias(c) for c in all_features.columns
]).collect()[0]
print("Null counts:")
for c in all_features.columns:
    print(f"  {c}: {null_counts[c]}")

# 5. Summary
print(f"✅ Unique (customer_id, feature_timestamp): {dup_count} duplicates")
print(f"✅ Row count: {feat_count} (matches observations: {obs_count})")
print(f"✅ Negative recency (leakage indicator): {negative_recency}")
print(f"✅ Features: {len(all_features.columns) - 2} feature columns + 2 key columns")

## Paso 3: Publicar Feature Table en Unity Catalog

Escribir `customer_features` como Delta table y registrarla con `FeatureEngineeringClient.create_table()`, declarando PK `customer_id + feature_timestamp` con columna TIMESERIES.

In [0]:
# ── Register Feature Table in Unity Catalog ──
# Idempotent: try create_table, fall back to write_table if table already exists

all_features_df = spark.sql("SELECT * FROM all_features")

try:
    fe.create_table(
        name=FEATURES_TABLE,
        primary_keys=["customer_id", "feature_timestamp"],
        timeseries_column="feature_timestamp",
        df=all_features_df,
        schema=all_features_df.schema,
        description="Customer features for repurchase propensity. One row per (customer_id, cutoff_ts). All features computed with events strictly before cutoff_ts.",
    )
    print(f"Feature Table created: {FEATURES_TABLE}")
except Exception as e:
    if "already exists" in str(e).lower():
        fe.write_table(name=FEATURES_TABLE, df=all_features_df, mode="overwrite")
        print(f"Feature Table overwritten: {FEATURES_TABLE}")
    else:
        raise

# Add column comments for discoverability
for spec in FEATURE_SPECS:
    comment = f"source: {spec['source']}, type: {spec['type']}, window: {spec['window']}, default: {spec['default']}"
    spark.sql(f"ALTER TABLE {FEATURES_TABLE} ALTER COLUMN {spec['name']} COMMENT '{comment}'")

# Also comment the observations table
spark.sql(f"COMMENT ON TABLE {OBSERVATIONS_TABLE} IS 'Customer observations for repurchase propensity. One row per (customer_id, cutoff_ts). Label = 1 if at least one completed order in [t, t+30d).'")

print(f"✅ Feature Table registered: {FEATURES_TABLE}")
print(f"   Primary keys: customer_id, feature_timestamp (TIMESERIES)")
print(f"   Features: {len(FEATURE_SPECS)}")

## Paso 4: Training set con FeatureLookup point-in-time

Usar `FeatureEngineeringClient.create_training_set` con `FeatureLookup` y `timestamp_lookup_key=cutoff_ts`. Esto garantiza que cada observación reciba el snapshot de features correspondiente a su corte, no el más reciente.

In [0]:
# ── Build training set with FeatureLookup ──
feature_lookups = [
    FeatureLookup(
        table_name=FEATURES_TABLE,
        lookup_key="customer_id",
        timestamp_lookup_key="cutoff_ts",
    )
]

training_set = fe.create_training_set(
    df=spark.table(OBSERVATIONS_TABLE).select("customer_id", "cutoff_ts", "repurchased_next_30d"),
    feature_lookups=feature_lookups,
    label="repurchased_next_30d",
)

training_df = training_set.load_df()
print(f"Training set: {training_df.count()} rows, {len(training_df.columns)} columns")
print(f"Columns: {training_df.columns}")
display(training_df.limit(10))

## Paso 5: Prueba automatizada contra leakage

Inyectar un evento posterior al cutoff y verificar que las features no cambian. Si el filtro temporal funciona correctamente, el evento inyectado no debe afectar las features del corte.

In [0]:
# ── Leakage test: inject future event, verify features don't change ──
test_row = spark.sql("""
    SELECT customer_id, feature_timestamp, days_since_last_order, orders_count_30d, orders_count_90d, avg_ticket_value_90d
    FROM all_features WHERE days_since_last_order < 9999 LIMIT 1
""").collect()[0]
test_customer = test_row["customer_id"]
test_cutoff = test_row["feature_timestamp"]
test_cutoff_str = test_cutoff.strftime("%Y-%m-%d %H:%M:%S")

print(f"Test customer: {test_customer}, cutoff: {test_cutoff_str}")
print(f"Original: days_since_last_order={test_row['days_since_last_order']}, orders_count_30d={test_row['orders_count_30d']}, orders_count_90d={test_row['orders_count_90d']}, avg_ticket_value_90d={test_row['avg_ticket_value_90d']}")

# Create orders source with an injected future event (5 days AFTER cutoff)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW injected_orders AS
    SELECT * FROM {CATALOG}.{SOURCE_SCHEMA}.orders
    UNION ALL
    SELECT
        'INJECTED_001' AS order_id,
        '{test_customer}' AS customer_id,
        date_add(timestamp('{test_cutoff_str}'), 5) AS order_ts,
        'web' AS channel,
        'completed' AS status,
        CAST(999.99 AS DOUBLE) AS total_amount,
        'USD' AS currency,
        false AS is_drift_period
""")

# Recompute orders-based features with injected data
recomputed = spark.sql(f"""
    SELECT
      obs.customer_id,
      obs.cutoff_ts AS feature_timestamp,
      COALESCE(
        DATEDIFF(obs.cutoff_ts, MAX(CASE WHEN o.order_ts < obs.cutoff_ts THEN o.order_ts END)),
        9999
      ) AS days_since_last_order,
      COUNT(CASE WHEN o.order_ts >= date_sub(obs.cutoff_ts, 30)
                 AND o.order_ts < obs.cutoff_ts THEN 1 END) AS orders_count_30d,
      COUNT(CASE WHEN o.order_ts >= date_sub(obs.cutoff_ts, 90)
                 AND o.order_ts < obs.cutoff_ts THEN 1 END) AS orders_count_90d,
      COALESCE(
        AVG(CASE WHEN o.order_ts >= date_sub(obs.cutoff_ts, 90)
                 AND o.order_ts < obs.cutoff_ts THEN o.total_amount END),
        0.0
      ) AS avg_ticket_value_90d
    FROM {OBSERVATIONS_TABLE} obs
    LEFT JOIN injected_orders o
      ON o.customer_id = obs.customer_id
    WHERE obs.customer_id = '{test_customer}' AND obs.cutoff_ts = timestamp('{test_cutoff_str}')
    GROUP BY obs.customer_id, obs.cutoff_ts
""").collect()[0]

# Assert no change
assert test_row["days_since_last_order"] == recomputed["days_since_last_order"], \
    f"LEAKAGE: days_since_last_order {test_row['days_since_last_order']} -> {recomputed['days_since_last_order']}"
assert test_row["orders_count_30d"] == recomputed["orders_count_30d"], \
    f"LEAKAGE: orders_count_30d {test_row['orders_count_30d']} -> {recomputed['orders_count_30d']}"
assert test_row["orders_count_90d"] == recomputed["orders_count_90d"], \
    f"LEAKAGE: orders_count_90d {test_row['orders_count_90d']} -> {recomputed['orders_count_90d']}"
assert test_row["avg_ticket_value_90d"] == recomputed["avg_ticket_value_90d"], \
    f"LEAKAGE: avg_ticket_value_90d {test_row['avg_ticket_value_90d']} -> {recomputed['avg_ticket_value_90d']}"

print(f"✅ LEAKAGE TEST PASSED: Future order 5 days after cutoff did not change any orders-based feature")
print(f"   Recomputed: days_since_last_order={recomputed['days_since_last_order']}, orders_count_30d={recomputed['orders_count_30d']}, orders_count_90d={recomputed['orders_count_90d']}, avg_ticket_value_90d={recomputed['avg_ticket_value_90d']}")

## Paso 6: Ejemplo de cliente en dos cortes

Mostrar un cliente en al menos dos fechas de corte con valores de features diferentes, demostrando que el sistema point-in-time entrega el snapshot correcto para cada corte.

In [0]:
# ── Example customer: show one customer at two cutoffs with different features ──
example_customer = spark.sql("""
    SELECT customer_id 
    FROM all_features 
    WHERE days_since_last_order < 9999 
    GROUP BY customer_id 
    HAVING COUNT(*) >= 2
    LIMIT 1
""").collect()[0]["customer_id"]

example_df = spark.sql(f"""
    SELECT 
        customer_id, 
        feature_timestamp,
        days_since_last_order,
        orders_count_30d,
        orders_count_90d,
        avg_ticket_value_90d,
        payment_failure_rate_90d,
        view_product_count_14d,
        search_count_14d,
        web_activity_count_14d,
        cart_abandonment_sessions_14d,
        favorite_product_sku,
        tenure_days,
        customer_segment
    FROM all_features 
    WHERE customer_id = '{example_customer}'
    ORDER BY feature_timestamp
""")

print(f"Example customer: {example_customer}")
print(f"Showing {example_df.count()} cutoffs:")
display(example_df)

## Cómo se evitó información del futuro

Toda feature se calcula con la condición `timestamp < cutoff_ts`. La etiqueta usa órdenes `completed` en `[cutoff_ts, cutoff_ts + 30d)`. El pago se filtra por `status_updated_at < cutoff_ts`. El training set se construye con `FeatureLookup` y `timestamp_lookup_key=cutoff_ts`, garantizando que cada observación recibe el snapshot de features correspondiente a su corte, no el más reciente. La prueba de leakage inyectó un evento futuro y verificó que las features no cambiaron.

## Evidencias a capturar

1. Catalog Explorer mostrando `customer_features` con PK y TIMESERIES
2. Catalog Explorer mostrando linaje de `customer_features` hacia `orders`, `payments`, `app_events`, `customers`
3. Output del training set con FeatureLookup
4. Output de la prueba de leakage (pasada)
5. Output del ejemplo de cliente en dos cortes
6. ADR-001 y ADR-002 visibles en el notebook
7. Tabla `customer_observations` con conteos y distribución de etiquetas
8. Comentarios de tabla y columnas visibles en Catalog Explorer